# ResNet18 ex novo v6.4 - Reject Option su split 80/20

Variante v6.4: v6.1 + LS=0.10 + Reject Option (Vento et al., S:11).

**Risultati v6.1 (0.9757 BA, 0.9784 TTA):**
- Two-phase training + Focal Loss ha dato il miglior modello
- Problemi residui: Plastic recall 0.91

**Novita v6.4:**
- **Label smoothing 0.10** (da 0.15): Focal Loss gestisce overconfidence, LS piu basso affina boundary.
- **Reject Option (Vento et al., S:11, L:reject_option)**:
  - ψ_a = max softmax confidence
  - ψ_b = top-2 margin
  - Val set splittato 50/50: val_cal per soglia, val_eval per metrica
  - Funzione di efficacia Ƥ con Cc=1, Cr=3, Ce=10
- **Split 80/20 singolo**: massimo training data = massima performance

Obiettivo: ridurre errori residui con reject option, minimizzando il reject rate.

## 1. Mount Drive e clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, datetime, subprocess
from getpass import getpass

REPO_OWNER = 'fabriziodrr'
REPO_NAME = 'ProjectML'
BASE_BRANCH = 'exp/resnet18-exnovo-v64'
RUN_ID = datetime.datetime.now().strftime('%Y%m%d-%H%M')
RUN_BRANCH = f'exp-resnet18-exnovo-v64-{RUN_ID}'
RUN_NAME = f'resnet18_exnovo_v64_{RUN_ID}'
REPO_PATH = f'/content/{REPO_NAME}'
PUBLIC_REPO_URL = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'

def sanitize(text):
    if 'AUTH_REPO_URL' in globals():
        text = text.replace(AUTH_REPO_URL, 'https://***@github.com/' + REPO_OWNER + '/' + REPO_NAME + '.git')
    if 'GITHUB_TOKEN' in globals() and GITHUB_TOKEN:
        text = text.replace(GITHUB_TOKEN, '***')
    return text

def run(cmd, cwd=None, check=True):
    printable = sanitize(' '.join(cmd))
    print('$', printable)
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout: print(sanitize(result.stdout))
    if result.stderr: print(sanitize(result.stderr))
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed: {printable}')
    return result

GITHUB_TOKEN = getpass('GitHub token personale con permessi repo: ')
AUTH_REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
if not os.path.exists(REPO_PATH):
    run(['git', 'clone', '--branch', 'main', AUTH_REPO_URL, REPO_PATH])
else:
    print('Repo gia presente.')
run(['git', 'remote', 'set-url', 'origin', AUTH_REPO_URL], cwd=REPO_PATH)
run(['git', 'pull', '--rebase', 'origin', 'main'], cwd=REPO_PATH)
run(['git', 'remote', 'set-url', 'origin', PUBLIC_REPO_URL], cwd=REPO_PATH)
branch_exists = subprocess.run(['git', 'rev-parse', '--verify', RUN_BRANCH], cwd=REPO_PATH, text=True, capture_output=True).returncode == 0
run(['git', 'switch', RUN_BRANCH if branch_exists else '-c', RUN_BRANCH] if branch_exists else ['git', 'switch', '-c', RUN_BRANCH], cwd=REPO_PATH)
os.chdir(REPO_PATH)
print('Run branch:', RUN_BRANCH)

## 2. Configurazione

In [ ]:
import json, random, shutil, zipfile
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

EXP_NAME = 'resnet18_exnovo_v64'
SEED = 42
NUM_CLASSES = 8
BATCH_SIZE = 32
NUM_WORKERS = 0
VAL_SIZE = 0.20
EPOCHS = 40
PATIENCE = 8
TWO_PHASE_FREEZE_EPOCHS = 5
FOCAL_GAMMA = 2.0
BACKBONE_LR = 1e-4
HEAD_LR = 5e-4
DROPOUT_P = 0.30
WEIGHT_DECAY = 2e-4
LABEL_SMOOTHING = 0.10
USE_AMP = True

Cc = 1.0; Cr = 3.0; Ce = 10.0

DRIVE_ZIP_PATH = '/content/drive/MyDrive/waste_type_identification.zip'
DRIVE_DATASET_DIR = '/content/drive/MyDrive/waste_type_identification'
LOCAL_DATASET_DIR = '/content/waste_type_identification'
RESULTS_DIR = Path(REPO_PATH) / 'results' / RUN_NAME
DRIVE_CHECKPOINT_DIR = Path('/content/drive/MyDrive/ProjectML_checkpoints') / RUN_NAME

CLASS_NAMES = ['Battery','Clothing','Glass','Metal','Organic','Papery','Plastic','Undifferentiated']
LABEL_MAP = {
    'battery':0, 'clothes':1, 'clothing':1, 'shoes':1,
    'brown':2, 'green':2, 'transparent':2, 'glass':2, 'metal':3, 'organic':4,
    'cardboard':5, 'paper':5, 'papery':5, 'plastic':6, 'undifferentiated':7,
}
IMG_EXTENSIONS = ('.jpg','.jpeg','.png','.bmp','.gif','.tiff','.webp')

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(SEED)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

## 3. Dataset e split 80/20 (train / val_cal / val_eval)

In [ ]:
local_dataset = Path(LOCAL_DATASET_DIR)
drive_dataset = Path(DRIVE_DATASET_DIR)
drive_zip = Path(DRIVE_ZIP_PATH)
if not local_dataset.exists():
    if drive_dataset.exists(): shutil.copytree(drive_dataset, local_dataset)
    elif drive_zip.exists():
        local_dataset.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(drive_zip, 'r') as zf: zf.extractall(local_dataset)
    else: raise FileNotFoundError('Dataset non trovato')
print('Dataset:', local_dataset)

In [ ]:
samples = []
for dirpath, _, filenames in os.walk(local_dataset, followlinks=False):
    folder = Path(dirpath).name.lower()
    if not filenames or folder not in LABEL_MAP: continue
    label = LABEL_MAP[folder]
    for fname in filenames:
        if fname.lower().endswith(IMG_EXTENSIONS):
            samples.append({'path':str(Path(dirpath)/fname), 'relative_path':str(Path(dirpath).relative_to(local_dataset)/fname),
                            'folder':folder, 'label':label, 'class_name':CLASS_NAMES[label]})

df = pd.DataFrame(samples).sort_values('relative_path').reset_index(drop=True)
print('Totale immagini:', len(df))
print(df['class_name'].value_counts())

# Split 80/20 (train / val_all)
splitter = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
train_idx, val_idx = next(splitter.split(df['relative_path'], df['label']))
df['split'] = 'train'
df.loc[val_idx, 'split'] = 'val'

# Split val 50/50 (val_cal / val_eval) stratified
val_mask = df['split'] == 'val'
val_df_all = df[val_mask]
splitter_val = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED+1)
cal_idx, eval_idx = next(splitter_val.split(val_df_all['relative_path'], val_df_all['label']))
val_indices = val_df_all.index
df.loc[val_indices[cal_idx], 'split'] = 'val_cal'
df.loc[val_indices[eval_idx], 'split'] = 'val_eval'

df[['relative_path','folder','label','class_name','split']].to_csv(RESULTS_DIR / 'split.csv', index=False)
print('\nSplit distribution:')
for s in ['train', 'val_cal', 'val_eval']:
    print(f'  {s}: {len(df[df["split"]==s])} samples')
print(pd.crosstab(df['class_name'], df['split']).reindex(CLASS_NAMES, fill_value=0))

## 4. Dataset, transform e DataLoader

In [ ]:
class WasteDataset(Dataset):
    def __init__(self, frame, transform=None):
        self.frame = frame.reset_index(drop=True); self.transform = transform
    def __len__(self): return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        if self.transform is not None: image = self.transform(image)
        return image, int(row['label'])

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.70, 1.0), ratio=(0.85, 1.18)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), shear=5),
    transforms.ColorJitter(brightness=0.30, contrast=0.25, saturation=0.20, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.10), ratio=(0.3, 3.3)),
])

val_transform = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_cal_df = df[df['split'] == 'val_cal'].reset_index(drop=True)
val_eval_df = df[df['split'] == 'val_eval'].reset_index(drop=True)

train_labels = train_df['label'].to_numpy()
class_sample_counts = np.bincount(train_labels)
sample_weights = (1.0 / class_sample_counts)[train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(WasteDataset(train_df, train_transform), batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
val_cal_loader = DataLoader(WasteDataset(val_cal_df, val_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
val_eval_loader = DataLoader(WasteDataset(val_eval_df, val_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

train_counts = train_df['label'].value_counts().reindex(range(NUM_CLASSES), fill_value=0).sort_index().to_numpy(dtype=np.float32)
class_weights = np.sqrt(train_counts.max() / np.maximum(train_counts, 1.0))
class_weights = class_weights / class_weights.mean()
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

weights_df = pd.DataFrame({'label': range(NUM_CLASSES), 'class_name': CLASS_NAMES, 'train_count': train_counts.astype(int), 'loss_weight': class_weights})
weights_df.to_csv(RESULTS_DIR / 'class_weights.csv', index=False)
print(weights_df)
print('WeightedRandomSampler attivo. Val split: cal/eval ready.')

## 5. Modello ResNet18 + Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss (S:09). FL(p_t) = -(1-p_t)^gamma * log(p_t)"""
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.weight = weight; self.gamma = gamma; self.label_smoothing = label_smoothing
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, label_smoothing=self.label_smoothing, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()

weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
model.fc = nn.Sequential(nn.Dropout(p=DROPOUT_P), nn.Linear(model.fc.in_features, NUM_CLASSES))
model = model.to(DEVICE)

criterion = FocalLoss(weight=class_weights_tensor, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
backbone_params = [p for n, p in model.named_parameters() if not n.startswith('fc.')]
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': BACKBONE_LR},
    {'params': model.fc.parameters(), 'lr': HEAD_LR},
], weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-6)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and DEVICE.type == 'cuda')

config = {
    'experiment': EXP_NAME, 'run_id': RUN_ID, 'run_name': RUN_NAME,
    'base_branch': BASE_BRANCH, 'run_branch': RUN_BRANCH,
    'model': 'ResNet18 pretrained ImageNet',
    'strategy': 'two_phase_freeze5ep_FocalLoss_gamma2_LS0.10_RejectOption_Vento_psiA_psiB_split8020_val5050',
    'seed': SEED, 'val_size': VAL_SIZE, 'batch_size': BATCH_SIZE, 'epochs': EPOCHS,
    'patience': PATIENCE, 'two_phase_freeze_epochs': TWO_PHASE_FREEZE_EPOCHS,
    'focal_gamma': FOCAL_GAMMA, 'label_smoothing': LABEL_SMOOTHING,
    'backbone_lr': BACKBONE_LR, 'head_lr': HEAD_LR, 'dropout_p': DROPOUT_P, 'weight_decay': WEIGHT_DECAY,
    'reject_costs': {'Cc': Cc, 'Cr': Cr, 'Ce': Ce},
    'reliability_functions': ['psi_a', 'psi_b'],
    'course_arguments': 'transfer+two-phase (S:12), FocalLoss (S:09), Reject Option Vento (S:11, L:reject_option), StratifiedShuffleSplit (L:01)',
    'results_dir': str(RESULTS_DIR),
}
with open(RESULTS_DIR / 'config.json', 'w') as f: json.dump(config, f, indent=2)
print(json.dumps(config, indent=2))

## 6. Two-phase training

In [ ]:
def freeze_backbone(model, freeze):
    for n, p in model.named_parameters():
        if not n.startswith('fc.'): p.requires_grad = not freeze

def run_epoch(model, loader, train):
    model.train(train)
    total_loss, all_preds, all_targets = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, targets in loader:
            images, targets = images.to(DEVICE, non_blocking=True), targets.to(DEVICE, non_blocking=True)
            if train: optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP and DEVICE.type == 'cuda'):
                logits = model(images); loss = criterion(logits, targets)
            if train: scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            total_loss += loss.item() * images.size(0)
            all_preds.append(logits.argmax(dim=1).detach().cpu())
            all_targets.append(targets.detach().cpu())
    yp = torch.cat(all_preds).numpy(); yt = torch.cat(all_targets).numpy()
    return total_loss/len(loader.dataset), float((yp==yt).mean()), float(balanced_accuracy_score(yt,yp)), yt, yp

best_ba, best_ep, no_impr = -1.0, -1, 0
history = []
best_ckpt = DRIVE_CHECKPOINT_DIR / 'resnet18_exnovo_v64_best.pth'

# Phase 1
freeze_backbone(model, True)
print(f'Phase 1: backbone frozen ({TWO_PHASE_FREEZE_EPOCHS} epochs)')
for ep in range(1, TWO_PHASE_FREEZE_EPOCHS+1):
    tl, ta, tba, _, _ = run_epoch(model, train_loader, True)
    vl, va, vba, _, _ = run_epoch(model, val_cal_loader, False)
    history.append({'epoch':ep,'phase':'freeze','train_loss':tl,'train_ba':tba,'val_loss':vl,'val_ba':vba})
    print(f'  E{ep:02d}[freeze]: tl={tl:.3f} tba={tba:.4f} vl={vl:.3f} vba={vba:.4f}')
    if vba > best_ba: best_ba, best_ep = vba, ep
    scheduler.step(vba)

# Phase 2
freeze_backbone(model, False)
no_impr = 0
print(f'Phase 2: backbone UNFROZEN')
for ep in range(TWO_PHASE_FREEZE_EPOCHS+1, EPOCHS+1):
    tl, ta, tba, _, _ = run_epoch(model, train_loader, True)
    vl, va, vba, _, _ = run_epoch(model, val_cal_loader, False)
    history.append({'epoch':ep,'phase':'finetune','train_loss':tl,'train_ba':tba,'val_loss':vl,'val_ba':vba})
    lr_str = f'lr=[{optimizer.param_groups[0]["lr"]:.2e},{optimizer.param_groups[1]["lr"]:.2e}]'
    print(f'  E{ep:02d}[finetune]: {lr_str} tl={tl:.3f} tba={tba:.4f} vl={vl:.3f} vba={vba:.4f}')
    if vba > best_ba:
        best_ba, best_ep, no_impr = vba, ep, 0
        torch.save({'model_state_dict':model.state_dict(),'best_epoch':ep,'best_val_ba':best_ba}, best_ckpt)
        print(f'    Saved best: BA={best_ba:.4f}')
    else:
        no_impr += 1
        if no_impr >= PATIENCE: print(f'Early stop epoch {ep}'); break
    scheduler.step(vba)

pd.DataFrame(history).to_csv(RESULTS_DIR/'history.csv', index=False)
summary = {'run_name':RUN_NAME,'run_branch':RUN_BRANCH,'results_dir':str(RESULTS_DIR),'best_epoch':best_ep,'best_val_bal_acc':best_ba,'checkpoint_path_drive':str(best_ckpt)}
with open(RESULTS_DIR/'summary.json','w') as f: json.dump(summary, f, indent=2)
print(f'\nTraining complete. Best val_cal BA: {best_ba:.4f} at epoch {best_ep}')

## 7. Reject Option (Vento et al., S:11) - Standard Evaluation

In [ ]:
checkpoint = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

def get_probs_and_labels(model, loader):
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, targets in loader:
            logits = model(images.to(DEVICE, non_blocking=True))
            all_probs.append(F.softmax(logits, dim=1).cpu().numpy())
            all_labels.append(targets.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)

def find_optimal_threshold(psi, labels, probs, name=''):
    preds = probs.argmax(axis=1)
    correct, error = (preds==labels), (preds!=labels)
    N = len(labels); Rc0, Re0 = correct.sum()/N, error.sum()/N
    best_P, best_tau, best_m = -np.inf, 0.0, None
    for tau in np.unique(psi):
        rejected = psi < tau; accepted = ~rejected
        if accepted.sum()==0: continue
        Rc, Re, Rr = correct[accepted].sum()/N, error[accepted].sum()/N, rejected.sum()/N
        P = Cc*(Rc-Rc0) - Ce*(Re-Re0) - Cr*Rr
        if P > best_P:
            best_P, best_tau = P, tau
            best_m = {'threshold':float(tau), 'P':float(P), 'reject_rate':float(Rr), 'coverage':float(1-Rr),
                      'accepted_acc':float(correct[accepted].sum()/accepted.sum()),
                      'accepted_ba':float(balanced_accuracy_score(labels[accepted], preds[accepted])),
                      'n_accepted':int(accepted.sum()), 'n_rejected':int(rejected.sum()),
                      'errors_avoided':int(error[rejected].sum())}
    print(f"  {name}: tau*={best_m['threshold']:.3f} | reject={best_m['reject_rate']:.1%} | "
          f"acc_BA={best_m['accepted_ba']:.4f} | acc={best_m['accepted_acc']:.4f} | err_avoided={best_m['errors_avoided']}")
    return best_tau, best_m

# Standard evaluation on val_cal (determine thresholds) and val_eval (evaluate)
print('--- Val_cal: get probs & find optimal thresholds ---')
probs_cal, labels_cal = get_probs_and_labels(model, val_cal_loader)
sorted_cal = np.sort(probs_cal, axis=1)[:, ::-1]
psi_a_cal, psi_b_cal = sorted_cal[:, 0], sorted_cal[:, 0] - sorted_cal[:, 1]

tau_a, _ = find_optimal_threshold(psi_a_cal, labels_cal, probs_cal, 'psi_a')
tau_b, _ = find_optimal_threshold(psi_b_cal, labels_cal, probs_cal, 'psi_b')

# Evaluate on val_eval
print('\n--- Val_eval: apply thresholds ---')
probs_eval, labels_eval = get_probs_and_labels(model, val_eval_loader)
sorted_eval = np.sort(probs_eval, axis=1)[:, ::-1]
preds_eval = probs_eval.argmax(axis=1)

# Baseline (no reject)
base_ba = float(balanced_accuracy_score(labels_eval, preds_eval))
base_acc = float((preds_eval == labels_eval).mean())
base_err = int((preds_eval != labels_eval).sum())
print(f'  No reject: BA={base_ba:.4f} Acc={base_acc:.4f} Errors={base_err}')

# With reject
reject_summary = {}
for name, tau, psi in [('psi_a', tau_a, sorted_eval[:,0]), ('psi_b', tau_b, sorted_eval[:,0]-sorted_eval[:,1])]:
    rejected = psi < tau; accepted = ~rejected
    ba = float(balanced_accuracy_score(labels_eval[accepted], preds_eval[accepted])) if accepted.sum()>0 else 0.0
    acc = float((preds_eval[accepted]==labels_eval[accepted]).mean()) if accepted.sum()>0 else 0.0
    cov = float(accepted.sum()/len(labels_eval))
    err_av = int((preds_eval[rejected]!=labels_eval[rejected]).sum())
    print(f'  {name}: tau={tau:.3f} | reject={1-cov:.1%} | BA={ba:.4f} | Acc={acc:.4f} | err_avoided={err_av}/{base_err}')
    reject_summary[name] = {'threshold':float(tau), 'reject_rate':float(1-cov), 'coverage':float(cov),
                            'accepted_ba':ba, 'accepted_acc':acc, 'errors_avoided':err_av}

with open(RESULTS_DIR/'reject_standard.json','w') as f: json.dump(reject_summary, f, indent=2)

## 8. TTA 10-crop + Reject Option

In [ ]:
crop_224 = transforms.FiveCrop(224)
tta_norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
to_ten = transforms.ToTensor()

def tta_probs(model, image):
    img256 = transforms.Resize(256)(image)
    crops = crop_224(img256)
    all_p = []
    for crop in crops:
        for flip in [False, True]:
            c = transforms.functional.hflip(crop) if flip else crop
            t = tta_norm(to_ten(c)).unsqueeze(0).to(DEVICE)
            with torch.no_grad(): all_p.append(F.softmax(model(t), dim=1))
    return torch.stack(all_p).mean(dim=0).cpu().numpy()

def tta_probs_and_labels(model, frame):
    probs, labels = [], []
    for idx in range(len(frame)):
        row = frame.iloc[idx]
        probs.append(tta_probs(model, Image.open(row['path']).convert('RGB')).flatten())
        labels.append(int(row['label']))
        if (idx+1)%500==0: print(f'  TTA: {idx+1}/{len(frame)}')
    return np.array(probs), np.array(labels)

print('TTA 10-crop on val_cal (threshold calibration)...')
probs_cal_tta, labels_cal_tta = tta_probs_and_labels(model, val_cal_df)
sorted_cal_tta = np.sort(probs_cal_tta, axis=1)[:, ::-1]

tau_a_tta, _ = find_optimal_threshold(sorted_cal_tta[:, 0], labels_cal_tta, probs_cal_tta, 'TTA psi_a')
tau_b_tta, _ = find_optimal_threshold(sorted_cal_tta[:, 0]-sorted_cal_tta[:, 1], labels_cal_tta, probs_cal_tta, 'TTA psi_b')

print('\nTTA 10-crop on val_eval (evaluation)...')
probs_eval_tta, labels_eval_tta = tta_probs_and_labels(model, val_eval_df)
sorted_eval_tta = np.sort(probs_eval_tta, axis=1)[:, ::-1]
preds_eval_tta = probs_eval_tta.argmax(axis=1)

base_ba_tta = float(balanced_accuracy_score(labels_eval_tta, preds_eval_tta))
base_acc_tta = float((preds_eval_tta == labels_eval_tta).mean())
base_err_tta = int((preds_eval_tta != labels_eval_tta).sum())
print(f'  TTA no reject: BA={base_ba_tta:.4f} Acc={base_acc_tta:.4f} Errors={base_err_tta}')

tta_reject_summary = {}
for name, tau, psi in [('psi_a', tau_a_tta, sorted_eval_tta[:,0]), ('psi_b', tau_b_tta, sorted_eval_tta[:,0]-sorted_eval_tta[:,1])]:
    rejected = psi < tau; accepted = ~rejected
    ba = float(balanced_accuracy_score(labels_eval_tta[accepted], preds_eval_tta[accepted])) if accepted.sum()>0 else 0.0
    acc = float((preds_eval_tta[accepted]==labels_eval_tta[accepted]).mean()) if accepted.sum()>0 else 0.0
    cov = float(accepted.sum()/len(labels_eval_tta))
    err_av = int((preds_eval_tta[rejected]!=labels_eval_tta[rejected]).sum())
    print(f'  TTA {name}: tau={tau:.3f} | reject={1-cov:.1%} | BA={ba:.4f} | Acc={acc:.4f} | err_avoided={err_av}/{base_err_tta}')
    tta_reject_summary[name] = {'threshold':float(tau), 'reject_rate':float(1-cov), 'coverage':float(cov),
                                'accepted_ba':ba, 'accepted_acc':acc, 'errors_avoided':err_av}

with open(RESULTS_DIR/'reject_tta.json','w') as f: json.dump(tta_reject_summary, f, indent=2)

## 9. Riepilogo finale, report e grafici

In [ ]:
print('\n========== FINAL SUMMARY v6.4 ==========')
print(f'Training BA (best on val_cal): {best_ba:.4f} at epoch {best_ep}')

print('\n--- Standard Evaluation (val_eval) ---')
print(f'No reject:    BA={base_ba:.4f} Acc={base_acc:.4f} Errors={base_err}')
for k, v in reject_summary.items():
    print(f'{k} reject: BA={v["accepted_ba"]:.4f} Acc={v["accepted_acc"]:.4f} reject={v["reject_rate"]:.1%} err_avoided={v["errors_avoided"]}')

print('\n--- TTA 10-crop Evaluation (val_eval) ---')
print(f'TTA no reject: BA={base_ba_tta:.4f} Acc={base_acc_tta:.4f} Errors={base_err_tta}')
for k, v in tta_reject_summary.items():
    print(f'TTA {k} reject: BA={v["accepted_ba"]:.4f} Acc={v["accepted_acc"]:.4f} reject={v["reject_rate"]:.1%} err_avoided={v["errors_avoided"]}')

# Classification report on val_eval (standard)
report = classification_report(labels_eval, preds_eval, target_names=CLASS_NAMES, zero_division=0)
cm = confusion_matrix(labels_eval, preds_eval, labels=list(range(NUM_CLASSES)))
with open(RESULTS_DIR/'classification_report.txt','w') as f: f.write(report)
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(RESULTS_DIR/'confusion_matrix.csv')
print('\n' + report)

# TTA report
tta_report = classification_report(labels_eval_tta, preds_eval_tta, target_names=CLASS_NAMES, zero_division=0)
tta_cm = confusion_matrix(labels_eval_tta, preds_eval_tta, labels=list(range(NUM_CLASSES)))
with open(RESULTS_DIR/'classification_report_tta.txt','w') as f: f.write(tta_report)
pd.DataFrame(tta_cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(RESULTS_DIR/'confusion_matrix_tta.csv')
print('--- TTA Report ---\n' + tta_report)

# Update summary
summary['val_eval_ba'] = base_ba
summary['val_eval_tta_ba'] = base_ba_tta
summary['reject_standard'] = reject_summary
summary['reject_tta'] = tta_reject_summary
with open(RESULTS_DIR/'summary.json','w') as f: json.dump(summary, f, indent=2)

# Plots
hdf = pd.read_csv(RESULTS_DIR/'history.csv')
fig, axes = plt.subplots(1,2,figsize=(12,4))
for phase, c in [('freeze','orange'),('finetune','blue')]:
    ph = hdf[hdf['phase']==phase]
    if len(ph)>0:
        axes[0].plot(ph['epoch'], ph['train_loss'], color=c, alpha=0.5, label=f'train {phase}')
        axes[0].plot(ph['epoch'], ph['val_loss'], color=c, ls='--', label=f'val {phase}')
        axes[1].plot(ph['epoch'], ph['train_ba'], color=c, alpha=0.5, label=f'train {phase}')
        axes[1].plot(ph['epoch'], ph['val_ba'], color=c, ls='--', label=f'val {phase}')
axes[0].set_title('v6.4 Loss'); axes[1].set_title('v6.4 BA'); axes[0].legend(); axes[1].legend()
plt.tight_layout(); plt.savefig(RESULTS_DIR/'training_curves.png', dpi=160); plt.show()

plt.figure(figsize=(8,7))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title('v6.4 Confusion Matrix'); plt.colorbar()
ticks = np.arange(NUM_CLASSES)
plt.xticks(ticks, CLASS_NAMES, rotation=45, ha='right'); plt.yticks(ticks, CLASS_NAMES)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES): plt.text(j,i,cm[i,j],ha='center',va='center',fontsize=8,color='white' if cm[i,j]>cm.max()/2 else 'black')
plt.tight_layout(); plt.savefig(RESULTS_DIR/'confusion_matrix.png', dpi=160); plt.show()

print('\nDone.')

## 10. Commit e push

In [ ]:
os.chdir(REPO_PATH)
run(['git', 'config', 'user.name', 'fabriziodrr'], cwd=REPO_PATH)
run(['git', 'config', 'user.email', 'fabriziodrr@users.noreply.github.com'], cwd=REPO_PATH)
run(['git', 'add', str(RESULTS_DIR.relative_to(REPO_PATH))], cwd=REPO_PATH)
status = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_PATH).decode().strip()
if not status:
    print('Nessuna modifica.')
else:
    run(['git', 'commit', '-m', f'Add ResNet18 exnovo v6.4 results {RUN_ID}'], cwd=REPO_PATH)
    auth_url = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
    try:
        run(['git', 'remote', 'set-url', 'origin', auth_url], cwd=REPO_PATH)
        run(['git', 'push', '-u', 'origin', RUN_BRANCH], cwd=REPO_PATH)
    finally:
        run(['git', 'remote', 'set-url', 'origin', PUBLIC_REPO_URL], cwd=REPO_PATH, check=False)
print('Done.')